# Depth Image and Point Cloud
### Depth Image
It is just a uint16 type image of size MxN where each pixel will have the depth value in mm from the camera.
### Point Cloud
This converts all the pixels in the depth image and will provide a 3D visualisation where each pixels are represented as a point in 3D space.

In [33]:
import cv2
import numpy as np
import open3d as o3d

### Conversion Math
If the camera intrinsics are $f_x$ (Focal Length in X axis), $f_y$ (Focal Length in Y axis), $c_x$, $c_y$ (Principal Points) and a pixel $(u,v)$ with a depth Z in meters then it is converted into a 3D point by\
$X = (u - c_x) . Z / f_x$\
$Y = (v - c_y) . Z / f_y$\
$Z = Z$

In [34]:
# Read the Image
depth_image = cv2.imread("/home/shri/development/Computer-Vision/tools/Synthetic_Depth_Image/depth_image.png", cv2.IMREAD_UNCHANGED)
if depth_image is None:
    raise RuntimeError("Could not read depth_image.png")

# Assign Values for Computing the Intrinsic Matrix
height, width = depth_image.shape
fx, fy = 615.0, 615.0
cx, cy = 320.0, 240.0

intrinsic = o3d.camera.PinholeCameraIntrinsic(width, height, fx, fy, cx, cy)

In [35]:
# convert the values into meters
depth_image_meters = (depth_image.astype(np.float32)) * 0.001

# Wrap it as an Open3D image and convert it into point cloud
o3d_depth = o3d.geometry.Image(depth_image_meters)
pcd = o3d.geometry.PointCloud.create_from_depth_image(
    o3d_depth,
    intrinsic, extrinsic=np.eye(4),
    depth_scale = 1.0, depth_trunc = 4.0,
    project_valid_depth_only=True)

In [36]:
# Visualise the Point Cloud
o3d.visualization.draw_geometries(
    [pcd],
    window_name="Point Cloud Visualiser",
    width=640,  # Window Size if needed
    height=480,
    point_show_normal=False
)

# Save Point Cloud
o3d.io.write_point_cloud("data/depth_pointcloud.pcd", pcd)

True

### Create and Visualize RGBD clouds


In [37]:
# Create a Sample Color Image and warp it as Open3D geometry image
color_image = np.ones((height, width, 3), dtype=np.uint8)
color_image[:, :, 2] = 100  # Change value of blue channel
o3d_color = o3d.geometry.Image(color_image)

# Create an RGBD Image
rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
    o3d_color, o3d_depth,
    depth_scale=1.0,
    depth_trunc=4.0,
    convert_rgb_to_intensity=False
)

In [38]:
# Create a RGB point cloud
pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)
o3d.visualization.draw_geometries(
    [pcd],
    width=640,  # Window Size if needed
    height=480)
o3d.io.write_point_cloud("data/rgbd_pointcloud.pcd", pcd)

True